### GOAL OF NOTEBOOK 04

We will:

* load processed RAG documents
* generate embeddings
* store vectors
* perform semantic search
* retrieve relevant airline complaints

This is the heart of RAG systems.

In [19]:
# Imports
import pandas as pd
from sentence_transformers import SentenceTransformer

### WHAT IS HAPPENING HERE?

We are using:

### Sentence Transformers

These models convert text into:

### vectors (embeddings)

Embeddings allow AI to:

* understand meaning
* compare similarity
* retrieve relevant knowledge

This is one of the MOST important concepts in modern AI.

In [20]:
# Load Processed Dataset
airline_df = pd.read_csv(
    "../data/processed/cleaned_airline_support.csv"
)

airline_df.head()

,airline,text,airline_sentiment,negativereason,clean_text,rag_document
0,Virgin America,@VirginAmerica plus you've added commercials t...,positive,No Issue,plus you ve added commercials to the experienc...,Airline: Virgin America\nSentiment: positive\n...
1,Virgin America,@VirginAmerica I didn't today... Must mean I n...,neutral,No Issue,i didn t today must mean i need to take anothe...,Airline: Virgin America\nSentiment: neutral\nI...
2,Virgin America,@VirginAmerica it's really aggressive to blast...,negative,Bad Flight,it s really aggressive to blast obnoxious ente...,Airline: Virgin America\nSentiment: negative\n...
3,Virgin America,@VirginAmerica and it's a really big bad thing...,negative,Can't Tell,and it s a really big bad thing about it,Airline: Virgin America\nSentiment: negative\n...
4,Virgin America,@VirginAmerica seriously would pay $30 a fligh...,negative,Can't Tell,seriously would pay 30 a flight for seats that...,Airline: Virgin America\nSentiment: negative\n...


In [21]:
# Load Embedding Model
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully!


In [22]:
# Generate First Embedding
sample_text = airline_df["rag_document"].iloc[0]

embedding = embedding_model.encode(sample_text)

print(type(embedding))
print(embedding.shape)

<class 'numpy.ndarray'>
(384,)


In [23]:
# Generate Embeddings for All Documents
documents = airline_df["rag_document"].tolist()

embeddings = embedding_model.encode(
    documents,
    show_progress_bar=True
)

print("Embeddings generated successfully!")

Batches:   0%|          | 0/448 [00:00<?, ?it/s]

Embeddings generated successfully!


In [24]:
# Check Embedding Shape
embeddings.shape

(14320, 384)

In [25]:
# Semantic Search Function
from sklearn.metrics.pairwise import cosine_similarity

In [26]:
# Create Search Function
def search_airline_knowledge(query, top_k=3):
    
    # encode query
    query_embedding = embedding_model.encode([query])
    
    # compute similarity
    similarities = cosine_similarity(
        query_embedding,
        embeddings
    )[0]
    
    # get top matches
    top_indices = similarities.argsort()[-top_k:][::-1]
    
    # print results
    for i, idx in enumerate(top_indices):
        
        print(f"\nRESULT {i+1}")
        print("="*80)
        
        print(airline_df.iloc[idx]["rag_document"])
        
        print(f"\nSimilarity Score: {similarities[idx]:.4f}")

In [27]:
# First Semantic Search
search_airline_knowledge(
    "My baggage was lost during my flight"
)


RESULT 1
Airline: US Airways
Sentiment: negative
Issue: Lost Luggage
Customer Message: but my bag is still missing

Similarity Score: 0.6719

RESULT 2
Airline: US Airways
Sentiment: negative
Issue: Lost Luggage
Customer Message: i ve done everything still no luck in finding my bag i don t understand how my bag could have been misplaced very upsetting

Similarity Score: 0.6667

RESULT 3
Airline: US Airways
Sentiment: negative
Issue: Lost Luggage
Customer Message: my flight was cancelled flighted saturday why was my luggage not at the airport today i m supposed to have it delivered pathetic

Similarity Score: 0.6649
